# Optimization Space: How Training Selects a Hypothesis

## 0. Learning objectives and lesson structure

The previous notebooks separated evidence and possibility:

$$
\mathcal{D}\quad\text{is the observed data,}
\qquad
\mathcal{H}\quad\text{is the set of candidate functions.}
$$

This notebook studies the optimisation ingredient:

$$
\boxed{\mathcal{H}} + \boxed{\mathcal{D}} + \boxed{\mathcal{O}} \rightarrow s.
$$

The central question is:

$$
\text{Given data }\mathcal{D}\text{ and candidates }\mathcal{H},\text{ how does training select one solution }s?
$$

A common answer is an optimisation rule that searches for parameters with small empirical loss:

$$
\widehat{\theta}
\approx
\arg\min_{\theta\in\Theta}
\widehat{R}_{\mathcal{D}}(h_{\theta}),
\qquad
\widehat{h}=h_{\widehat{\theta}}.
$$

By the end, students should be able to:
- define an objective function from data, hypotheses, loss, and regularisation;
- distinguish empirical risk from population risk;
- explain gradient descent as an iterative search procedure;
- describe how learning rate and curvature affect optimisation behaviour;
- explain why stochastic gradients introduce noise and bias in the training path;
- interpret regularisation, early stopping, and optimiser choices as selection pressures;
- connect optimisation paths in parameter space to selected functions in hypothesis space.

In [ ]:
# Setup cell for the notebook.
#
# This cell should eventually:
# - import numpy, matplotlib, and IPython display utilities;
# - optionally import ipywidgets for sliders and preset buttons;
# - create shared plotting helpers for objective curves, contours, and optimisation paths;
# - define a fixed random seed for repeatable classroom demonstrations;
# - prepare a simple regression dataset that later cells can reuse.
#
# Interaction notes:
# - Run this once at the start of the notebook.
# - Keep all examples small enough that path animations and slider updates are immediate.
# - Make the selected objective, optimiser, and parameter path visible in every example.

## 1. From a hypothesis space to one selected solution

A hypothesis space contains many possible functions:

$$
\mathcal{H}=\{h_{\theta}:\theta\in\Theta\}.
$$

Optimisation supplies a rule for selecting one of them. For supervised data

$$
\mathcal{D}=\{(x_i,y_i)\}_{i=1}^{n},
$$

an empirical objective often has the form

$$
J(\theta)
=
\widehat{R}_{\mathcal{D}}(h_{\theta})+\lambda\Omega(\theta),
$$

where

$$
\widehat{R}_{\mathcal{D}}(h_{\theta})
=
\frac{1}{n}\sum_{i=1}^{n}\ell(h_{\theta}(x_i),y_i)
$$

measures fit to observed data, and \(\Omega\) encodes an additional preference or penalty.

The selected parameter is

$$
\widehat{\theta}\in\operatorname*{arg\,min}_{\theta\in\Theta}J(\theta),
$$

when the minimum can be found exactly. In practice, training often returns an approximate solution after a finite sequence of updates:

$$
\theta_0,\theta_1,\ldots,\theta_T,
\qquad
s=h_{\theta_T}.
$$

This makes \(\mathcal{O}\) more than a loss function. It includes the optimiser, initialisation, learning-rate schedule, stopping rule, and any regularisation.

In [ ]:
# Example cell: selecting one function from a fixed hypothesis space.
#
# This cell should eventually:
# - generate a fixed small regression dataset D;
# - define a simple hypothesis space, such as h_theta(x) = theta_0 + theta_1*x;
# - compute the empirical objective J(theta) over a grid of theta_0 and theta_1 values;
# - plot several candidate lines from H and highlight the one selected by the objective;
# - make explicit that D and H stay fixed while O selects a solution.
#
# Suggested interaction:
# - Use a dropdown for the objective: squared error, absolute error, or regularised loss.
# - Let students click or slide candidate theta values and inspect their objective value.
# - Ask: did we change the data, the hypothesis space, or the optimisation rule?

## 2. Loss functions define what counts as a mistake

The loss function maps a prediction and target to a penalty:

$$
\ell(\widehat{y},y)\geq 0.
$$

For regression, squared loss is

$$
\ell_2(\widehat{y},y)=(\widehat{y}-y)^2,
$$

and absolute loss is

$$
\ell_1(\widehat{y},y)=|\widehat{y}-y|.
$$

With residual \(r=\widehat{y}-y\), these penalties behave differently:

$$
\ell_2(r)=r^2,
\qquad
\ell_1(r)=|r|.
$$

Squared loss emphasises large residuals because its derivative grows with residual size:

$$
\frac{d}{dr}r^2=2r.
$$

For binary classification with \(y\in\{0,1\}\) and predicted probability \(p_{\theta}(x)\), cross-entropy loss is

$$
\ell_{\mathrm{CE}}(p,y)
=-y\log p-(1-y)\log(1-p).
$$

Changing \(\ell\) changes the training objective and can change the selected solution even when \(\mathcal{D}\) and \(\mathcal{H}\) are fixed.

In [ ]:
# Example cell: compare losses on the same residuals and dataset.
#
# This cell should eventually:
# - plot squared loss, absolute loss, and optionally Huber loss as functions of residual r;
# - fit the same simple model under two different losses or display precomputed fits;
# - include one outlier so students can see how loss choice changes the selected line;
# - report each candidate model's total empirical loss under each loss function.
#
# Suggested interaction:
# - Use a checkbox to add/remove an outlier.
# - Use a dropdown to choose the active loss and update the selected model.
# - Ask students which points dominate the objective under squared loss.

## 3. Objective landscapes in parameter space

For a linear model

$$
h_{\theta}(x)=\theta_0+\theta_1x,
$$

with squared loss, the empirical objective is

$$
J(\theta_0,\theta_1)
=
\frac{1}{n}\sum_{i=1}^{n}(\theta_0+\theta_1x_i-y_i)^2.
$$

In matrix form, with design matrix \(X\), target vector \(y\), and parameter vector \(\theta\),

$$
J(\theta)=\frac{1}{n}\|X\theta-y\|_2^2.
$$

This objective is a convex quadratic. Its gradient is

$$
\nabla J(\theta)=\frac{2}{n}X^{\top}(X\theta-y),
$$

and the Hessian is

$$
\nabla^2J(\theta)=\frac{2}{n}X^{\top}X.
$$

The condition number of \(X^{\top}X\) affects the shape of the contours. Well-conditioned objectives have rounder contours; poorly conditioned objectives have long narrow valleys that are harder to traverse.

In [ ]:
# Example cell: visualise an empirical objective landscape.
#
# This cell should eventually:
# - build a small linear regression problem with parameters theta_0 and theta_1;
# - evaluate J(theta_0, theta_1) over a grid;
# - show contour lines of the objective in parameter space;
# - mark the least-squares solution and a few alternative parameter choices;
# - optionally compare a well-conditioned dataset and a poorly conditioned dataset.
#
# Suggested interaction:
# - Use a slider to move a point in parameter space and update the fitted line.
# - Add a toggle for feature scaling to show how scaling changes contour geometry.
# - Ask students to connect contour shape to expected optimiser behaviour.

## 4. Gradient descent as iterative search

Gradient descent updates parameters by moving against the objective gradient:

$$
\theta_{t+1}=\theta_t-\eta\nabla J(\theta_t),
$$

where \(\eta>0\) is the learning rate.

For one-dimensional quadratic objective

$$
J(\theta)=\frac{1}{2}a(\theta-\theta^{\star})^2,
\qquad a>0,
$$

we have

$$
\nabla J(\theta)=a(\theta-\theta^{\star}),
$$

so the update becomes

$$
\theta_{t+1}-\theta^{\star}
=(1-\eta a)(\theta_t-\theta^{\star}).
$$

The iteration converges when

$$
|1-\eta a|<1,
\qquad\text{equivalently}\qquad
0<\eta<\frac{2}{a}.
$$

This simple case shows why learning rate matters. Too small means slow progress; too large can oscillate or diverge.

In [ ]:
# Example cell: gradient descent on a one-dimensional quadratic.
#
# This cell should eventually:
# - define J(theta) = 0.5*a*(theta - theta_star)^2;
# - run gradient descent for several learning rates eta;
# - plot the objective curve and mark theta_t at each iteration;
# - plot theta_t over time to show convergence, oscillation, or divergence;
# - print the convergence condition 0 < eta < 2/a for the selected curvature a.
#
# Suggested interaction:
# - Use sliders for curvature a, learning rate eta, and initial theta_0.
# - Let students predict whether a selected eta will converge before running the path.
# - Use a step button or animation to make the iterative nature visible.

## 5. Learning rate, curvature, and conditioning

For a multivariate quadratic objective

$$
J(\theta)=\frac{1}{2}(\theta-\theta^{\star})^{\top}A(\theta-\theta^{\star}),
$$

where \(A\) is symmetric positive definite, the gradient is

$$
\nabla J(\theta)=A(\theta-\theta^{\star}).
$$

Gradient descent updates errors according to

$$
\theta_{t+1}-\theta^{\star}
=(I-\eta A)(\theta_t-\theta^{\star}).
$$

Convergence depends on the eigenvalues of \(A\):

$$
0<\eta<\frac{2}{\lambda_{\max}(A)}.
$$

The condition number

$$
\kappa(A)=\frac{\lambda_{\max}(A)}{\lambda_{\min}(A)}
$$

controls how stretched the landscape is. Large \(\kappa\) often leads to zig-zagging paths and slow progress along shallow directions.

Feature scaling can change the optimisation geometry without changing the underlying prediction problem.

In [ ]:
# Example cell: gradient descent on a two-dimensional contour plot.
#
# This cell should eventually:
# - define a two-dimensional quadratic with adjustable eigenvalues;
# - plot contours of J(theta_1, theta_2);
# - run gradient descent from a shared starting point;
# - overlay the optimisation path on the contour plot;
# - compare paths for well-conditioned and ill-conditioned objectives.
#
# Suggested interaction:
# - Use sliders for lambda_min, lambda_max, eta, and theta_0.
# - Add a feature-scaling toggle in the linear-regression version of the example.
# - Ask students why the path may zig-zag even though every step reduces the objective.

## 6. Stochastic optimisation and minibatches

Full-batch gradient descent uses every data point in each update:

$$
\nabla J(\theta)
=
\frac{1}{n}\sum_{i=1}^{n}\nabla_{\theta}\ell(h_{\theta}(x_i),y_i).
$$

Minibatch stochastic gradient descent uses a subset \(B_t\subset\{1,\ldots,n\}\):

$$
g_t
=
\frac{1}{|B_t|}\sum_{i\in B_t}\nabla_{\theta}\ell(h_{\theta}(x_i),y_i),
$$

and updates

$$
\theta_{t+1}=\theta_t-\eta g_t.
$$

If the minibatch is sampled uniformly, then \(g_t\) is an unbiased estimator of the full gradient:

$$
\mathbb{E}[g_t\mid\theta_t]=\nabla J(\theta_t).
$$

But it has variance:

$$
\operatorname{Var}(g_t\mid\theta_t)>0.
$$

This noise can make training paths less smooth, help escape some regions, and change which solution is selected.

In [ ]:
# Example cell: full-batch versus minibatch optimisation paths.
#
# This cell should eventually:
# - create a regression dataset with enough points for minibatches;
# - run full-batch gradient descent and minibatch SGD on the same objective;
# - plot objective value versus update number for both methods;
# - plot parameter paths to show smooth versus noisy movement;
# - optionally repeat SGD with several random seeds to show path variability.
#
# Suggested interaction:
# - Use controls for batch size, learning rate, and random seed.
# - Ask students what changes when batch size decreases.
# - Emphasise that SGD noise is part of O, not part of D or H.

## 7. Regularisation as an explicit selection pressure

Regularisation modifies the objective by adding a penalty:

$$
J_{\lambda}(\theta)
=
\widehat{R}_{\mathcal{D}}(h_{\theta})+\lambda\Omega(\theta),
\qquad \lambda\geq 0.
$$

For ridge regression,

$$
\Omega(\theta)=\|\theta\|_2^2,
$$

so the objective is

$$
J_{\lambda}(\theta)
=
\frac{1}{n}\|X\theta-y\|_2^2+\lambda\|\theta\|_2^2.
$$

For linear least squares, the ridge solution is

$$
\widehat{\theta}_{\lambda}
=(X^{\top}X+n\lambda I)^{-1}X^{\top}y.
$$

The penalty changes which good-fitting function is preferred. With many hypotheses that fit \(\mathcal{D}\), regularisation can select the one with smaller norm, smoother behaviour, or another desired property.

In [ ]:
# Example cell: regularisation changes the selected solution.
#
# This cell should eventually:
# - fit polynomial regression models with ridge penalties over several lambda values;
# - plot the fitted functions and show how larger lambda dampens coefficients;
# - report training error, validation/test error, and parameter norm ||theta||_2;
# - show that lambda changes O while D and H can remain fixed.
#
# Suggested interaction:
# - Use a log-scale slider for lambda.
# - Let students compare training fit against function smoothness and extrapolation.
# - Ask which solution they would deploy and what preference their choice encodes.

## 8. Non-convexity, initialisation, and multiple solutions

Deep-network objectives are generally non-convex:

$$
J(\theta)=\frac{1}{n}\sum_{i=1}^{n}\ell(h_{\theta}(x_i),y_i)
$$

may contain many local minima, saddle points, flat regions, and symmetries. Training starts from an initial parameter vector

$$
\theta_0\sim P_{\mathrm{init}},
$$

then follows an optimiser-defined path. Different initialisations can lead to different parameter solutions:

$$
\theta_T^{(1)}\neq\theta_T^{(2)}.
$$

These parameter solutions may still produce similar functions, or they may generalise differently:

$$
h_{\theta_T^{(1)}}\approx h_{\theta_T^{(2)}}
\quad\text{on training data, but not necessarily elsewhere.}
$$

Optimisation therefore participates in model selection. The selected solution depends on the objective, starting point, optimiser dynamics, stochasticity, and stopping time.

In [ ]:
# Example cell: different initialisations can select different solutions.
#
# This cell should eventually:
# - define a simple non-convex toy objective, such as a double-well or rippled quadratic;
# - run gradient descent from several initial theta_0 values;
# - plot paths that converge to different local minima;
# - optionally compare with a tiny neural-network example if dependencies are available;
# - record final objective values and final parameter values in a small table.
#
# Suggested interaction:
# - Use clickable starting points or an initialisation slider.
# - Let students add multiple starts and observe the basin of attraction.
# - Ask whether the objective value alone is enough to decide which solution is best.

## 9. Early stopping and implicit bias

Training returns a finite-time solution:

$$
s=h_{\theta_T},
$$

not necessarily the exact minimiser of the empirical objective. The stopping time \(T\) can act like regularisation.

For iterative optimisation,

$$
\theta_0,\theta_1,\ldots,\theta_T,
$$

early iterates may fit broad structure before later iterates fit noise. A validation-based stopping rule is often written as

$$
T^{\star}
=\arg\min_{t\in\{0,\ldots,T_{\max}\}}
\widehat{R}_{\mathrm{val}}(h_{\theta_t}).
$$

Optimisers can also impose implicit bias. Even without an explicit penalty \(\Omega\), the path may favour some solutions over others:

$$
\widehat{\theta}_{\mathrm{opt}}
\in
\left\{\theta : \widehat{R}_{\mathcal{D}}(h_{\theta})\approx 0\right\}
$$

but not all near-interpolating parameters are equally likely to be reached. The optimiser, learning rate schedule, batch size, and initialisation all shape this preference.

In [ ]:
# Example cell: early stopping as a selection rule.
#
# This cell should eventually:
# - train or simulate a flexible model over many iterations;
# - track training error and validation error at each iteration;
# - plot the selected early-stopping iteration T_star;
# - show the fitted function at early, best-validation, and late iterations;
# - connect stopping time to the final selected solution s = h_{theta_T}.
#
# Suggested interaction:
# - Use a slider for iteration t to scrub through the training trajectory.
# - Add a vertical marker at the validation-selected stopping time.
# - Ask students to identify when the model starts fitting noise rather than structure.

## 10. Summary and link to investigation space

Optimisation converts candidates into a selected solution:

$$
\mathcal{O}: (\mathcal{H},\mathcal{D})\mapsto s.
$$

In practice, \(\mathcal{O}\) includes:

$$
\text{loss} + \text{regularisation} + \text{initialisation} + \text{update rule} + \text{stochasticity} + \text{stopping rule}.
$$

The full workshop frame is now visible:

$$
\boxed{\mathcal{H}} + \boxed{\mathcal{D}} + \boxed{\mathcal{O}} \rightarrow s.
$$

Two learners can share the same data and hypothesis space but select different solutions because their optimisation procedures differ:

$$
(\mathcal{H},\mathcal{D},\mathcal{O}_1)\rightarrow s_1,
\qquad
(\mathcal{H},\mathcal{D},\mathcal{O}_2)\rightarrow s_2.
$$

The next notebook can ask how we investigate a trained solution: what evidence supports it, where it fails, and which part of \(\mathcal{H}+\mathcal{D}+\mathcal{O}\) explains the behaviour we observe.

In [ ]:
# Optional wrap-up cell.
#
# This cell should eventually:
# - display a compact summary table with columns for D, H, O, and selected solution s;
# - list the optimisation choices made in the notebook examples;
# - let students classify observed behaviour as caused by data, hypothesis space,
#   objective/loss, optimiser dynamics, regularisation, or stopping time;
# - prepare a handoff question for the next investigation notebook.
#
# Suggested interaction:
# - Use this as an exit ticket or small-group discussion prompt.
# - Keep the focus on diagnosis: which ingredient changed, and how did the solution change?